# 08 · Modem Lab · BPSK to 64-QAM

**Local software lesson · offline examples by default**

**What is added:** a Python reference chain and interactive browser lab for BPSK, QPSK, 16-QAM and 64-QAM.

**Why it is useful:** adjust noise, carrier offset, timing and multipath, then see what synchronization, hard/soft decisions and FEC do to the same known frame.

**How it works:** the seeded Python model creates a framed waveform, adds channel impairments, acquires the burst and compares decoded bits with the known payload. The local browser page displays raw and recovered samples, EVM, CRC/lock state and error counts.

```text
Python TX reference → seeded software channel → Python RX reference → measured BER / EVM
```

The modem ladder and synthetic RF-link examples are explicitly **Python simulation**. The original ESP32 has transmitted framed BPSK recovered by an independent receiver; a captured frame has separately passed through the physical FPGA symbol/FEC block, and a same-capture paired test verified the existing FPGA FIR II IP. The Python carrier/timing recovery does not imply that equivalent raw-waveform receiver blocks exist in FPGA.

The optional RF Link Lab page starts idle. Only its explicit **Capture RF** button requests hardware capture, which requires the installed v4/v5 TX probe (QPSK: v5) and active IQB1 runtime hub. Starting the viewer or running the offline cells does not start RF, and the notebook does not flash or program hardware.

## 1 · Set up the offline model

Run from the repository's Atum A3 IQ Demos environment. This lesson needs the existing NumPy and Matplotlib packages. It does not need Quartus, Questa, a board, ESP32 firmware or a network connection.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cyberether/modem_lab.py").exists())
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "cyberether"))
from lessons import lan_ip
from modem_reference import run_experiment
print("Example root:", ROOT)
print("Backend: Python reference simulation; no hardware source")

## 2 · Compare the modulation ladder

All four runs use the same SNR, seed and payload length. Higher-order modes carry more coded bits per symbol, while their ideal points sit closer together at the same average symbol energy. The following plots are synthetic simulation results.

In [ ]:
from notebook_progress import progress
with progress("Run Python example / render graphs"):
    import matplotlib.pyplot as plt
    from notebook_progress import progress
    MODES = ("BPSK", "QPSK", "16QAM", "64QAM")
    ladder = {}
    with progress("Simulate modulation ladder"):
        for mode in MODES:
            ladder[mode] = run_experiment(modulation=mode, snr_db=18, cfo_hz=1500,
                timing_offset=.25, multipath=False, seed=7, payload_bits=2048,
                decode_mode="soft", coded=True, sample_rate=4_000_000, sps=4)
    fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
    for ax, mode in zip(axes, MODES):
        plot = ladder[mode]["plots"]
        ideal = plot["ideal_xy"]
        received = plot["recovered_xy"]
        ax.scatter([p[0] for p in ideal], [p[1] for p in ideal], s=9, c="#71869d", marker="s", label="ideal")
        ax.scatter([p[0] for p in received], [p[1] for p in received], s=7, c="#18aeb2", alpha=.45, label="recovered")
        ax.set(title=mode, xlabel="I", ylabel="Q", aspect="equal")
        ax.grid(alpha=.2)
    axes[0].legend(loc="lower left", fontsize=8)
    fig.suptitle("Python simulation · recovered symbols and ideal reference")
    fig.tight_layout()
    plt.show()
    for mode, result in ladder.items():
        m = result["metrics"]
        print(f"{mode:6} EVM={m['evm_percent']:.2f}%  pre-FEC={m['pre_fec_errors']}/{m['pre_fec_bits']}  soft payload errors={m['soft_fec_errors']}/{m['post_fec_bits']}")

## 3 · Change one impairment at a time

This sweep keeps modulation, seed and payload fixed while changing carrier offset. Each result is a separate seeded simulation. Acquisition failure, a bad header or CRC failure is a real model outcome and stays visible in the metrics.

In [ ]:
from notebook_progress import progress
with progress("Run Python example / render graphs"):
    carrier_sweep = []
    with progress("Measure carrier-offset sweep"):
        for offset in (0, 1500, 5000, 10000):
            result = run_experiment(modulation="64QAM", snr_db=18, cfo_hz=offset,
                timing_offset=.25, multipath=False, seed=21, payload_bits=4096,
                decode_mode="soft", coded=True, sample_rate=4_000_000, sps=4)
            carrier_sweep.append((offset, result["metrics"]))
    for offset, m in carrier_sweep:
        print(f"CFO {offset:5} Hz · lock={m['locked']} · EVM={m['evm_percent']:.2f}% · hard errors={m['hard_fec_errors']}/{m['post_fec_bits']} · soft errors={m['soft_fec_errors']}/{m['post_fec_bits']} · CRC={m['crc_ok']}")

## 4 · Accumulate a 64-QAM BER curve

This longer run aggregates 49 independent, seeded 2048-bit payloads at each Es/N0 point. It compares hard and soft Viterbi input over the same software channel and reports actual error counts and tested-bit denominators. The reference BER helper excludes frames that fail acquisition/header decoding and reports those separately. It labels every row as **Python simulation**. This takes longer than the single-frame controls, so the cell shows elapsed progress and the current SNR; reduce `BER_SEEDS` for a quick preview. A zero-error observation is shown as a separate annotated marker and count, not as BER=0 on the logarithmic axis.

In [ ]:
from notebook_progress import progress
with progress("Run Python example / render graphs"):
    from modem_reference import ber_sweep

    BER_SNR_DB = (12, 16, 18, 22)
    BER_SEEDS = tuple(range(49))
    BER_PAYLOAD_BITS = 2048
    ber_rows = []
    with progress("Accumulate 64-QAM BER curve") as activity:
        for snr in BER_SNR_DB:
            activity.message(f"{len(BER_SEEDS)} independent frames · {BER_PAYLOAD_BITS} payload bits/frame · Es/N0 {snr} dB")
            result = ber_sweep(modulation="64QAM", snr_values=(snr,), seeds=BER_SEEDS,
                payload_bits=BER_PAYLOAD_BITS, cfo_hz=1500, timing_offset=.25,
                multipath=False, coded=True, decode_mode="soft",
                sample_rate=4_000_000, sps=4)
            ber_rows.extend(result["rows"])

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for key, label, color, marker in (
        ("hard_fec_ber", "Hard-decision Viterbi", "#d38b2c", "o"),
        ("soft_fec_ber", "Soft-decision Viterbi", "#168f9b", "s"),
    ):
        measured = [row for row in ber_rows if row[key] is not None and row[key] > 0]
        ax.plot([r["snr_db"] for r in measured], [r[key] for r in measured],
                color=color, marker=marker, linewidth=2, label=label)
        zeros = [row for row in ber_rows if row[key] == 0]
        if zeros:
            # The marker lives in the bottom annotation strip, not at an invented BER value.
            ax.scatter([r["snr_db"] for r in zeros], [.055] * len(zeros),
                       transform=ax.get_xaxis_transform(), marker="v", s=50,
                       facecolors="none", edgecolors=color, linewidths=1.5,
                       label=f"{label}: zero observed (annotation only)")
            for row in zeros:
                ax.annotate(f"0/{row['post_fec_bits']}",
                            xy=(row["snr_db"], .055), xycoords=ax.get_xaxis_transform(),
                            xytext=(0, 8), textcoords="offset points", ha="center",
                            color=color, fontsize=8, rotation=90)
    ax.set_yscale("log")
    ax.set_xlabel("Es/N0 (dB)")
    ax.set_ylabel("Observed payload BER (nonzero counts)")
    ax.set_title("Python simulation · 64-QAM hard vs soft decoding")
    ax.grid(True, which="both", alpha=.25)
    ax.legend(fontsize=8)
    ax.text(.01, .015, "Hollow triangles and 0/N labels mark zero observed errors; they are not BER coordinates.",
            transform=ax.transAxes, fontsize=8, color="#555")
    fig.tight_layout()
    plt.show()
    print("Source: Python simulation · Es/N0 sweep · same seeded frames for hard/soft decoding")
    for row in ber_rows:
        total = row["post_fec_bits"]
        def count(key):
            errors = row[key]
            return f"{errors}/{total} = {errors/total:.6g}" if errors else f"0/{total} observed; BER not plotted"
        print(f"{row['snr_db']:>2.0f} dB · frames {row['frames']} · failures {row['frame_failures']} · hard {count('hard_fec_errors')} · soft {count('soft_fec_errors')}")

## 5 · Open the interactive browser lab

This starts a Python server bound to `0.0.0.0` so a browser on your LAN can reach the notebook host. The printed URL uses `lan_ip()`; replace `HOST` below with an address reachable from your browser if route discovery chooses the wrong interface. The optional FPGA action remains off unless you explicitly set `ENABLE_FPGA_ACTION=True`; even then its separate button requires a click, a matching checkpoint 06 manifest/image and a free Nios/JTAG owner. Stop the process by interrupting this cell or use the cleanup cell. Python plots remain labeled simulation; FPGA results use a separate badge and evidence.

In [ ]:
import subprocess
LAB_LOG = ROOT / "cyberether/run/modem-lab.log"
LAB_LOG.parent.mkdir(parents=True, exist_ok=True)
LAB_PORT = 8769
HOST = lan_ip()  # Set to a reachable host/container LAN address if needed.
ENABLE_FPGA_ACTION = False  # Opt in explicitly; still requires a manual click and checkpoint guard.
lab_log = LAB_LOG.open("ab")
args = [sys.executable, str(ROOT / "cyberether/modem_lab.py"), "--host", "0.0.0.0", "--port", str(LAB_PORT)]
if ENABLE_FPGA_ACTION:
    args.append("--enable-fpga-action")
lab_process = subprocess.Popen(args, stdout=lab_log, stderr=subprocess.STDOUT, cwd=ROOT)
LAB_URL = f"http://{HOST}:{LAB_PORT}/"
print("Modem Lab:", LAB_URL)
print("Server PID:", lab_process.pid, "· bind: 0.0.0.0 · source: Python simulation")

### Saved browser example

This static screenshot was captured from the working local page. Values are one seeded Python simulation run; the banner records the original-ESP32 arbitrary-IQ TX qualification boundary, separate from the measured phase-pulse BPSK link below.

![Modem Lab browser example](screenshots/08_modem_lab.png)

[Open the full-resolution screenshot](screenshots/08_modem_lab.png).

## 6 · Original ESP32 RF Link Lab

The framed phase-BPSK contract is **64 symbols**: 16 marker symbols carrying `0x0651` MSB first, 44 rate-1/2 K=7 coded symbols from payload byte + CRC-8 + six zero tail bits, then four zero guard symbols. The CRC uses polynomial `0x07` and initial value `0x00`; the convolutional generators are `(171,133)` octal. At the default 8 µs per symbol, the frame takes 512 µs. A 16,380-sample capture at 16 MS/s spans 1.02375 ms, nominally about two frame periods. TX scheduling jitter or missed slots can delay or restart a frame, so a capture is not guaranteed to contain a complete frame.

**Bring-up:** corrected v4 firmware passed its normal `READY` startup. **Measured phase calibration:** with the original ESP32 at 160 MHz, extension step 3 and 670 cycles, the median phase change was −180.22° with second circular moment 0.992. The half-width control measured −90.01°, and CW produced zero events. The nominal 682-cycle full pulse measured −183.11° with moment 0.656, failing the strict concentration criterion; the checker therefore defaults to the measured 670-cycle profile at 160 MHz.

**Measured RF frames:** `rf-bpsk-001` recovered four CRC-valid `0xA5` frames (1/176 coded errors, 0/32 payload errors); `002` recovered sixteen `0x3C` frames (19/704, 0/128); `003` recovered four `0x00` frames (0/176, 0/32); and `004` recovered four `0xFF` frames (0/176, 0/32). All 28 frames passed CRC: **20/1232 pre-FEC coded errors and 0/224 payload errors**. An independent review redecoded all raw captures without expected-payload input and verified their hashes. Known payloads were used only after host acquisition for BER comparison. These are observed sparse samples, not evidence of zero true BER or delivery between capture windows.

**Separate FPGA FIR II test:** two paired captures processed 32,760 complex samples through the existing checkpoint 03 16-tap Hamming Q15 FIR II with nearest rounding. FPGA output matched the independent integer reference exactly (maximum error 0); processing used 114,756 cycles (2.29512 ms at 50 MHz) per 16,380-sample burst. Raw and filtered paths each recovered two CRC-valid frames with 0/88 coded errors. This is separate FIR-IP verification; those frames are not included in the 28 direct-RF totals, and no BER improvement is claimed.

```text
                           ┌→ Raw CI8 snapshot ─────────┐
Same ESP32 RF snapshot ────┤                             ├→ Host sync / BPSK / K7 FEC / CRC
                           └→ Existing FPGA FIR II ────┘
                                16-tap Hamming Q15 · nearest
```

The offline preview below separately creates synthetic clean and FEC-challenge IQ, then runs the host reference and plots its actual returned spectrum, marker correlation, raw IQ and recovered symbols. It remains explicitly **offline synthetic simulation** and does not access a radio or FPGA. Raw-IQ synchronization remains host processing.

In [ ]:
from notebook_progress import progress
with progress("Run Python example / render graphs"):
    import matplotlib.pyplot as plt
    from notebook_progress import progress
    from rf_bpsk import synthesize, receive

    RF_BPSK_PAYLOAD = 0xA5
    rf_examples = {}
    with progress("Generate offline framed-BPSK examples"):
        rf_examples["clean"] = receive(
            synthesize(RF_BPSK_PAYLOAD, seed=7), expected_payload=RF_BPSK_PAYLOAD)
        rf_examples["FEC challenge"] = receive(
            synthesize(RF_BPSK_PAYLOAD, seed=7, coded_flips=(5, 20)),
            expected_payload=RF_BPSK_PAYLOAD)

    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    clean_plots = rf_examples["clean"]["plots"]
    challenge_plots = rf_examples["FEC challenge"]["plots"]
    axes[0, 0].plot(clean_plots["spectrum"]["x"], clean_plots["spectrum"]["y"])
    axes[0, 0].set(title="Synthetic clean · spectrum", xlabel="Frequency (Hz)", ylabel="Relative power (dB)")
    axes[0, 1].plot(clean_plots["correlation"]["x"], clean_plots["correlation"]["y"])
    axes[0, 1].set(title="Synthetic clean · marker correlation", xlabel="Sample", ylabel="Score")
    axes[1, 0].scatter(clean_plots["raw_iq"]["i"], clean_plots["raw_iq"]["q"], s=3, alpha=.25)
    axes[1, 0].set(title="Synthetic clean · raw IQ", xlabel="I", ylabel="Q")
    axes[1, 1].scatter(challenge_plots["constellation"]["i"], challenge_plots["constellation"]["q"], s=12)
    axes[1, 1].set(title="Synthetic FEC challenge · recovered symbols", xlabel="I", ylabel="Q")
    for ax in axes.flat:
        ax.grid(alpha=.2)
        if ax in (axes[1, 0], axes[1, 1]):
            ax.set_aspect("equal", adjustable="datalim")
    fig.suptitle("OFFLINE SYNTHETIC · host receiver output · no RF capture")
    fig.tight_layout()
    plt.show()
    for label, result in rf_examples.items():
        print(f"OFFLINE SYNTHETIC · {label}: acquired={result['acquired_frames']} "
              f"CRC-valid={result['valid_frames']} sync-failures={result['acquisition_failures']} "
              f"coded={result['coded_errors']}/{result['coded_bits']} "
              f"payload={result['payload_errors']}/{result['payload_bits']} "
              f"CRC-failures={result['crc_failures']}")

### BER qualification · hard versus soft decisions

What is added: repeatable AWGN trials with 16-byte payloads, paired hard/soft Viterbi decoding and explicit error denominators. Soft metrics preserve confidence that a hard bit discards. Both decoders see the **same** received samples.

This Python symbol simulation assumes perfect timing/carrier recovery. It does not transmit, acquire RF, or execute FPGA DSP. Eb/N0 includes payload and CRC information bits, with finite encoder-tail cost; preamble/guards are excluded. A zero-error observation is plotted as an upper-bound triangle, never as a logarithm of zero. Bit bounds assume independent errors; post-FEC errors correlate, so independent-frame bounds are also returned. Actual saved RF results remain a separate audit.

```text
Random bytes → CRC → K7 FEC → BPSK → AWGN ┬→ hard bits → Viterbi → count
                                         └→ soft metrics → Viterbi → count
```

Use `scripts/check_rf_ber.py` or the page's **BER comparison · simulation** button to repeat the larger 1,000-frame/point experiment. [Detailed controls and evidence](../docs/BER_QUALIFICATION.md).

In [ ]:
from notebook_progress import progress
with progress("Run Python example / render graphs"):
    from ber_qualification import run_sweep
    from notebook_progress import progress
    import matplotlib.pyplot as plt

    with progress("Compare hard and soft FEC · offline AWGN"):
        qualification = run_sweep([-2, 0, 2, 4, 6], frames=1000,
                                  payload_bytes=16, seed=20261021)
    fig, ax = plt.subplots(figsize=(10, 5))
    for name, color in [("hard", "#e8a63d"), ("soft", "#159dab")]:
        rows = [point[name]["payload"] for point in qualification["points"]]
        x = [point["ebn0_db"] for point in qualification["points"]]
        y = [row["rate"] if row["errors"] else row["zero_error_upper"] for row in rows]
        ax.semilogy(x, y, "-", color=color, label=name + " Viterbi")
        for db, value, row in zip(x, y, rows):
            ax.plot(db, value, "o" if row["errors"] else "v", color=color)
        print(name, [(row["errors"], row["trials"]) for row in rows])
        print("95% zero-error frame bounds:",
              [point[name]["frames"]["zero_error_upper"] for point in qualification["points"]])
    ax.set(xlabel="Eb/N0 per encoder information bit (dB)", ylabel="Payload BER / zero-error upper bound",
           title="SYNTHETIC AWGN · same received symbols · perfect synchronization")
    ax.grid(which="both", alpha=.25)
    ax.legend()
    plt.show()

### Saved BER comparison page

The upper graph is the offline AWGN experiment. The separate RF audit below it preserves real capture counts and receiver identity; it does not establish a hard/soft RF advantage when both decoders succeed.

![RF Link Lab BER experiment and separate physical evidence](screenshots/08_rf_link_lab_ber.png)

### QPSK · two coded bits per recovered symbol

The original ESP32 can send a compact QPSK phase-pulse frame: 32 known training states, 22 coded payload/CRC symbols and 10 guards. Host synchronization estimates carrier, timing and phase from training. The FPGA can separately demap/decode recovered symbols; it does not yet synchronize raw IQ.

Two limits apply: the capture window must contain the 54 training and coded symbols needed for acquisition and decode, and the transmitter must meet its symbol deadlines. The graphic below estimates only the first limit. It assumes a uniformly unknown frame phase and an ideal usable window; it is theory, not a measured phase distribution. It does not model DSP errors or explain individual acquisition failures.

The latest v6 RF run at 12 µs acquired and CRC-validated 34 frames in 64 captures. A separate v6 run at 10 µs acquired 30/32 captures and CRC-validated all 30, with 0/1320 coded and 0/240 payload errors. At 8 µs, 32 captures yielded no acquisition because TX completed zero frames: that is a timing failure, not zero BER. These outcomes are examples, not a fit to the bars: capture phase is request-coupled and was not measured as uniform. The 10 µs result does not disprove the uniform-phase geometry model or establish its assumed distribution. Keep acquisition and CRC failures in their own denominators.

The offline example below uses a longer two-frame software record. Real ESP32/FPGA snapshots remain 16,380 samples. Known payloads enter error measurement only after acquisition. [QPSK controls, results and limits](../docs/QPSK_QUALIFICATION.md).

In [ ]:
from notebook_progress import progress
with progress("Estimate sparse-capture coverage · theory only"):
    import numpy as np
    import matplotlib.pyplot as plt

    capture_us = 16380 / 16e6 * 1e6
    symbol_periods_us = np.array([8, 9, 10, 12], dtype=float)
    frame_symbols = 64
    training_symbols, coded_symbols, guard_symbols = 32, 22, 10
    needed_symbols = training_symbols + coded_symbols
    coverage = np.clip((capture_us - needed_symbols * symbol_periods_us) /
                       (frame_symbols * symbol_periods_us), 0, 1)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    bars = ax.bar([f"{period:g} µs" for period in symbol_periods_us],
                  coverage * 100, color="#159dab")
    ax.bar_label(bars, labels=[f"{value:.1f}%" for value in coverage * 100], padding=3)
    ax.set(ylim=(0, 110), ylabel="Idealized full-window coverage (%)",
           xlabel="QPSK symbol period",
           title="THEORY ONLY · uniform unknown frame phase")
    ax.grid(axis="y", alpha=.25)
    ax.text(.99, .97, f"W = {capture_us:.2f} µs · {needed_symbols} useful / {frame_symbols} total",
            transform=ax.transAxes, ha="right", va="top", fontsize=8)
    plt.show()
    print("Model only: uniform relative phase; no DSP errors or measured phase distribution.")


In [ ]:
from notebook_progress import progress
with progress("Run Python example / render graphs"):
    from rf_qpsk import synthesize as qpsk_samples, receive as qpsk_receive
    import matplotlib.pyplot as plt
    from notebook_progress import progress

    with progress("Recover a synthetic QPSK frame · no hardware"):
        qpsk_result = qpsk_receive(
            qpsk_samples(0x3C, symbol_us=12, samples=24576, seed=17),
            symbol_us=12, expected_payload=0x3C)
    points = qpsk_result["plots"]["constellation"]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(points["i"], points["q"], s=18, label="Recovered sample averages")
    for i, q, label in [(1, 0, "00"), (0, -1, "01"), (-1, 0, "11"), (0, 1, "10")]:
        ax.scatter([i], [q], facecolors="none", edgecolors="gray", s=160)
        ax.annotate(label, (i, q), xytext=(10, 10), textcoords="offset points")
    ax.set(xlabel="I", ylabel="Q", title="SYNTHETIC QPSK · circles are ideal Gray references")
    ax.set_aspect("equal")
    ax.grid(alpha=.25)
    ax.legend()
    plt.show()
    print("CRC-valid:", qpsk_result["valid_frames"], "acquisition failures:", qpsk_result["acquisition_failures"])
    print("Coded errors:", qpsk_result["coded_errors"], "/", qpsk_result["coded_bits"])
    print("Payload errors:", qpsk_result["payload_errors"], "/", qpsk_result["payload_bits"])

### Saved physical QPSK page

This example uses actual RF samples. The FPGA replay panel records a separate transaction on measured symbols, including the explicit −135° convention rotation. It preserves received noise and amplitude; it does not substitute ideal constellation points.

![Real QPSK capture and separate FPGA decode](screenshots/08_rf_link_lab_qpsk.png)

### Optional · start the RF Link Lab viewer

The service opens no serial or JTAG device and does not start RF. Its page is initially idle; changing the payload, symbol period or pulse width only changes requested settings. The explicit **Capture RF** button starts a bounded capture and requires the compatible transmitter firmware (BPSK: v4/v5; QPSK: v5) plus the IQB1 FPGA runtime hub. The offline synthetic preview above is separate and never controls that button.

The first manual FPGA replay used frame 0 from `rf-bpsk-001`: the FPGA block ID was `0x4d440601`, coded errors were 1/44, decoded payload errors 0/8 and CRC passed. Decode time was 3,144 cycles (62.88 µs at 50 MHz); the JTAG transaction took 86.402 seconds. The separate `scripts/check_rf_bpsk_fpga.py` operation sends host-recovered symbols to checkpoint 06's soft demapper and K=7 FEC. It does not program the image or synchronize raw IQ; it uses Nios/JTAG. The normal checkpoint 03 image was restored after replay.

The FIR test is also a separate manual operation, `scripts/check_rf_bpsk_fir.py`, using the already available checkpoint 03-compatible FIR image. It verifies paired raw and FIR output from the same RF snapshot against the exact integer reference; FIR filtering does not move synchronization or FEC into FPGA.

In [ ]:
import subprocess

START_RF_LINK_LAB = False  # Opt in to starting the idle viewer; this does not start RF.
if START_RF_LINK_LAB:
    result = subprocess.run(
        [sys.executable, str(ROOT / "scripts/start_rf_link_lab.py"), "start"],
        cwd=ROOT, check=True, capture_output=True, text=True)
    print(result.stdout.strip())
    print("RF remains idle until Capture RF is clicked in the page.")
else:
    print("RF Link Lab viewer skipped. Set START_RF_LINK_LAB=True to start its idle service.")

### CyberEther · measured and synthetic RF Link Lab views

The synthetic page screenshot preserves the software-only Clean reference and FEC challenge controls. The physical RF screenshot shows four direct RF bursts, the paired raw/FIR spectrum, FPGA FIR reference checks, and the separate symbol/FEC replay. Its summary panel reports the `0xA5` run. The RF page is idle in the screenshot; only **Capture RF** requests a bounded RF job.

Latest v6 QPSK GUI check: 10 µs symbols, payload `0x3C`, four captures contained five acquired frames: three passed CRC and two failed. The plot retains a CRC-valid capture; the counters include failures. The acknowledgement reports the actual TX job, channel 8, attenuation code 120 and 500 ms bound. RF power is uncalibrated. This is a finite functional check, not a BER qualification.

![Live v6 QPSK and actual transmitter acknowledgement](screenshots/08_rf_link_lab_v6.png)

![RF Link Lab synthetic example](screenshots/08_rf_link_lab_synthetic.png)

[Open the full-resolution synthetic screenshot](screenshots/08_rf_link_lab_synthetic.png).

![RF Link Lab physical capture, FIR check and FPGA replay](screenshots/08_rf_link_lab_rf.png)

[Open the full-resolution physical RF screenshot](screenshots/08_rf_link_lab_rf.png).

## 7 · FPGA path and its current boundary

Checkpoint 06 implements symbol-rate encoding/mapping, max-log soft demapping and Viterbi FEC. Run the actual Questa assertions, then optionally generate the native VDS project and build a SOF using the executable cells below. Each action is controlled by a default-false switch so **Run All** stays offline. Generation/build use the project progress helper and log output. They do not program a board or flash a radio. The saved PNG is an actual Quartus VDS GUI export from checkpoint 06. Python waveform shaping, burst acquisition, carrier/timing recovery and equalization are not implemented in this RTL. Future work is integrating the full raw-waveform receiver in FPGA, evaluating higher-throughput SPI transport and extending beyond BPSK.

### Saved native Quartus VDS GUI export

![Checkpoint 06 modem block design exported from the actual VDS GUI](diagrams/08_vds_gui.png)

This is the native design GUI export, not a Python topology drawing. The executable generate cell can recreate the project and export a fresh diagram locally.

In [ ]:
from IPython.display import Image, display
from pathlib import Path
from simulate_modem import simulate
from modem_fpga import build as build_modem

# Enable only the action you intend to run. Simulation/build/generation never program hardware.
RUN_QUESTA_SIMULATION = False
GENERATE_NATIVE_VDS = False
BUILD_MODEM_SOF = False

if RUN_QUESTA_SIMULATION:
    rtl_result = simulate(gui=False)  # Actual RTL assertions; no FPGA/JTAG/serial access.
    print("RTL simulation:", rtl_result["status"])
if GENERATE_NATIVE_VDS:
    modem_qpf = build_modem(generate_only=True)  # Tool invocation shows progress and writes a log.
    print("Generated project:", modem_qpf)
if BUILD_MODEM_SOF:
    modem_qpf = build_modem(generate_only=False)  # Quartus compile only; no board programming.
    print("Built SOF:", modem_qpf.with_suffix(".sof"))

saved_diagram = ROOT / "notebooks/diagrams/08_vds_gui.png"
display(Image(filename=str(saved_diagram)))
print("Saved actual VDS export:", saved_diagram)

### Optional · program checkpoint 06

This replaces the currently loaded FPGA image with the modem checkpoint. It checks the build fingerprint and SOF SHA, stops only this checkout's notebook-owned demo, refuses remaining serial/JTAG owners and uses the explicit Atum JTAG cable setting. Do not run this while another acquisition service or design owner is active. The cell is separate from generation/simulation and defaults off; it does not flash or reset an ESP32. Review the compiled image and its resource/timing reports before explicitly running this cell.

In [ ]:
from modem_fpga import program

RUN_PROGRAM_MODEM = False  # Explicitly set True and run this cell to load checkpoint 06.
if RUN_PROGRAM_MODEM:
    active_modem = program()  # Hash/source guard + device-owner guard + progress.
    print("Loaded:", active_modem["stage_relative"], "· hardware qualification remains separate.")
else:
    print("Programming skipped. Set RUN_PROGRAM_MODEM=True only for the explicit checkpoint 06 load.")

## Cleanup

Stop the local page and release its port when finished. The process is separate from FPGA acquisition services.

In [ ]:
if "lab_process" in globals() and lab_process.poll() is None:
    lab_process.terminate()
    lab_process.wait(timeout=5)
    lab_log.close()
    print("Modem Lab stopped.")
else:
    print("No Modem Lab server is running from this kernel.")

## FPGA 64-QAM · constellation and 100k-bit BER sweep

**Added:** actual FPGA encoding/mapping, noisy symbol replay, soft demapping and Viterbi, with constellation and error counters together.

**Useful:** compare coded-bit BER with decoded-payload BER over at least 100,000 payload bits per noise level.

**Implemented:** checkpoint06 RTL performs modem DSP; Nios provides a seeded payload and approximate Gaussian noise, moves the IQ through Avalon and independently audits FPGA counters. Python verifies a saved frame from every noise level and draws the plots. No RF TX or raw-waveform synchronization occurs. The default test covers 100,096 payload bits at each of 12, 16, 20 dB Es/N0. See [the FPGA BER guide](../docs/FPGA_64QAM_BER.md).

In [ ]:
RUN_FPGA_64QAM_SWEEP = False
if RUN_FPGA_64QAM_SWEEP:
    sweep_output = ROOT / "cyberether/run" / ("qam64-notebook-" + str(time.time_ns()))
    with progress("FPGA 64-QAM BER sweep · 100k payload bits per level · no RF TX"):
        subprocess.run([sys.executable, str(ROOT / "scripts/check_qam64_sweep.py"),
                        "--output", str(sweep_output)], cwd=ROOT, check=True)
    proof = json.loads((sweep_output / "proof.json").read_text())
    for row in proof["rows"]:
        print(row["esn0_db"], "dB:", row["raw_errors"], "/", row["coded_bits"],
              "coded errors →", row["decoded_errors"], "/", row["payload_bits"], "payload errors")
else:
    print("Skipped. Explicitly build/program checkpoint06 before enabling the hardware sweep.")

In [ ]:
START_FPGA_64QAM_PAGE = False
if START_FPGA_64QAM_PAGE:
    subprocess.run([sys.executable, str(ROOT / "scripts/start_qam64_lab.py"), "start"], cwd=ROOT, check=True)
else:
    print("Viewer launch skipped. It starts idle and never programs or transmits.")

### Measured FPGA 64-QAM example

The physical Agilex 3 sweep tested 100,096 payload bits per noise level. At 12 dB: 23,559/204,884 coded errors → 176/100,096 payload errors. At 16 and 20 dB: 10,121 and 1,679 coded errors respectively, with no payload errors observed. These are FPGA symbol-replay measurements with Nios-generated approximate Gaussian noise, not RF captures. Each plotted constellation shows the last audited frame at that noise level, while the BER table includes all 391 frames; that individual frame may be error-free even if the aggregate has errors.

![FPGA 64-QAM constellation and BER](screenshots/08_fpga_64qam_ber.png)

For the compact TX/RX presentation, open `/poster` on the same port (for example `http://HOST_IP:8773/poster`). It shows noisy received samples above before/after-FEC overlays at all three measured levels. Green rings reconstruct the same coded symbols from actual FPGA-decoded bits, using host display-only re-encoding/remapping; received IQ is not denoised. Frame error counts and full-sweep BER are kept separate. This page is read-only and starts no acquisition. Its download link provides the saved example PNG.

![FPGA 64-QAM TX/RX social layout](screenshots/08_fpga_64qam_social.png)
